# BNBUSDT — DCA Buy-the-Bear Final (Progressive Reinvestment)

**Parameters:** Start 1%, +1% per red-month buy, cap 40%, reset after each 70% ATH sell.

Strategy: Buy $10 BNB + reinvest X% of USDT reserve every red monthly candle. Sell 70% at every new ATH.

In [ ]:
import pandas as pd, requests, time, numpy as np, matplotlib.pyplot as plt, matplotlib.dates as mdates
from datetime import datetime, timezone
pd.set_option('display.max_columns', None); pd.set_option('display.width', 200); pd.set_option('display.float_format', lambda x: '%.6f' % x)

In [ ]:
def fetch_monthly_klines(symbol='BNBUSDT', limit=500):
    url = 'https://api.binance.com/api/v3/klines'
    all_klines = []; start_time = 0
    while True:
        params = {'symbol': symbol, 'interval': '1M', 'startTime': start_time, 'limit': limit}
        resp = requests.get(url, params=params); data = resp.json()
        if not data or len(data) == 0: break
        all_klines.extend(data)
        print(f'Fetched {len(data)} months, up to {datetime.fromtimestamp(data[-1][0]/1000, tz=timezone.utc).strftime("%Y-%m")}')
        if len(data) < limit: break
        start_time = data[-1][0] + 1; time.sleep(0.1)
    return all_klines
klines = fetch_monthly_klines()

In [ ]:
cols = ['timestamp','open','high','low','close','volume','close_time','quote_vol','trades','taker_buy_base','taker_buy_quote','ignore']
df = pd.DataFrame(klines, columns=cols); df['date'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)
for c in ['open','high','low','close','volume']: df[c] = df[c].astype(float)
df = df[['date','open','high','low','close','volume']].copy().sort_values('date').reset_index(drop=True)
print(f'{df["date"].min().strftime("%Y-%m")} → {df["date"].max().strftime("%Y-%m")} ({len(df)} months)')

In [ ]:
# Parameters (optimum for BNB from grid search)
START_PCT = 1.0
INCREMENT = 1.0
CAP = 40.0

coin_held = 0.0; usdt_reserve = 0.0; total_invested = 0.0; highest_close = 0.0; reinvest_pct = START_PCT
records = []

for i, row in df.iterrows():
    close = row['close']; month = row['date']; action = 'nothing'

    if close < row['open']:
        extra = usdt_reserve * (reinvest_pct / 100.0) if usdt_reserve > 0 else 0.0
        total_usdt = 10.0 + extra
        coin_held += total_usdt / close
        total_invested += 10.0
        usdt_reserve -= extra
        action = f'buy ${total_usdt:.1f} @ {close:.1f} (${10:.0f}+{reinvest_pct:.0f}%={extra:.1f})'

    prev_highest = highest_close
    if close > highest_close: highest_close = close

    if coin_held > 0.000001 and close > prev_highest:
        sell_coin = coin_held * 0.7
        sell_usdt = sell_coin * close
        coin_held -= sell_coin
        usdt_reserve += sell_usdt
        reinvest_pct = START_PCT
        act = f'sell 70% @ {close:.1f} (-{sell_coin:.6f} BNB, +{sell_usdt:.1f} USDT)'
        action = f'{action} | {act}' if 'buy' in action else act
    else:
        if reinvest_pct < CAP:
            reinvest_pct = min(CAP, reinvest_pct + INCREMENT)

    portfolio_value = coin_held * close + usdt_reserve
    coin_value = coin_held * close

    records.append({'date': month, 'close': close, 'action': action, 'reinvest_pct': reinvest_pct,
                    'coin_held': coin_held, 'coin_value': coin_value, 'usdt_reserve': usdt_reserve,
                    'total_invested': total_invested, 'portfolio_value': portfolio_value})

results = pd.DataFrame(records)
print(f'Total months: {len(results)}')
print(f'Trades with action: {len(results[results["action"] != "nothing"])}')
buys_ct = len([r for r in records if 'buy' in r['action']])
sells_ct = len([r for r in records if 'sell' in r['action']])
print(f'Buy months: {buys_ct}')
print(f'Sell months: {sells_ct}')

In [ ]:
# Metrics computation

final = results.iloc[-1]
last_close = final['close']
total_pnl = final['portfolio_value'] - final['total_invested']
ret_pct = (final['portfolio_value'] / final['total_invested'] - 1) * 100

eq = results['portfolio_value']
running_max = eq.cummax()
dd_raw = (running_max - eq) / running_max
max_dd = dd_raw.max() * 100

monthly_returns = eq.pct_change().dropna()
monthly_returns = monthly_returns[monthly_returns.index >= 12]
sharpe = (monthly_returns.mean() / monthly_returns.std()) * np.sqrt(12) if monthly_returns.std() > 0 else 0.0

pos_sum = monthly_returns[monthly_returns > 0].sum()
neg_sum = monthly_returns[monthly_returns < 0].abs().sum()
pf = pos_sum / neg_sum if neg_sum > 0 else float('inf')

annualized_ret = (1 + ret_pct / 100) ** (12 / len(df)) - 1
calmar = annualized_ret / (max_dd / 100) if max_dd > 0 else 0

buys = [r for r in records if 'buy' in r['action']]
sells = [r for r in records if 'sell' in r['action']]

print('='*65)
print('PORTFOLIO SUMMARY — BNBUSDT Progressive (Optimum)')
print('='*65)
print(f'Parameters:            start={START_PCT}%, +{INCREMENT}%/buy, cap={CAP}%')
print(f'Total invested:        {final["total_invested"]:.2f} USDT')
print(f'BNB held:              {final["coin_held"]:.8f} BNB')
print(f'BNB value:             {final["coin_value"]:.2f} USDT (at {last_close:.2f})')
print(f'USDT reserve:          {final["usdt_reserve"]:.2f} USDT')
print(f'Portfolio value:       {final["portfolio_value"]:.2f} USDT')
print(f'Total P&L:             {total_pnl:.2f} USDT')
print(f'Return:                {ret_pct:.2f}%')
print(f'Max drawdown:          {max_dd:.2f}%')
print(f'Sharpe ratio:          {sharpe:.2f}')
print(f'Profit factor:         {pf:.2f}')
print(f'Calmar ratio:          {calmar:.2f}')
print(f'Months:                {len(results)}')
print(f'Buy months:            {len(buys)}')
print(f'Sell months:           {len(sells)}')

print()
print('CANDIDATE COMPARISON ON BNB:')
print('-'*60)
for lbl, st, inc, cp in [('1%+1%->40% (best Calmar)', 1,1,40),
                           ('1%+1%->30% (best Sharpe)', 1,1,30),
                           ('3%+1%->50% (high R low DD)', 3,1,50),
                           ('1%+2%->50% (BTC opt)', 1,2,50)]:
    c = 0.0; u = 0.0; inv = 0.0; hc = 0.0; rp = st
    for _, row in df.iterrows():
        c2 = row['close']
        if c2 < row['open']:
            ext = u * (rp/100.0) if u > 0 else 0.0
            c += (10.0 + ext) / c2; inv += 10.0; u -= ext
        ph = hc
        if c2 > hc: hc = c2
        if c > 0.000001 and c2 > ph:
            s = c * 0.7; u += s * c2; c -= s; rp = st
        elif rp < cp: rp = min(cp, rp + inc)
    fv = c * df.iloc[-1]['close'] + u
    r = (fv / inv - 1) * 100
    eq2 = pd.Series([0]*12+[1])
    dd2 = 0
    print(f'  {lbl:>30s}  R={r:6.1f}%')

In [ ]:
# Chart 1: Portfolio growth with buy/sell markers + drawdown

fig = plt.figure(figsize=(14, 10))
gs = fig.add_gridspec(4, 1, height_ratios=[3, 1, 1.2, 1.2])

ax1 = fig.add_subplot(gs[0])
ax1.fill_between(results['date'], results['total_invested'], results['portfolio_value'],
                 where=results['portfolio_value'] >= results['total_invested'],
                 color='green', alpha=0.12, label='Profit')
ax1.fill_between(results['date'], results['portfolio_value'], results['total_invested'],
                 where=results['portfolio_value'] < results['total_invested'],
                 color='red', alpha=0.12, label='Loss')
ax1.plot(results['date'], results['total_invested'], color='gray', linewidth=1.5, linestyle='--', label='Total Invested')
ax1.plot(results['date'], results['portfolio_value'], color='#2563eb', linewidth=2, label='Portfolio Value')

buys_df = results[results['action'].str.contains('buy', na=False)]
sells_df = results[results['action'].str.contains('sell', na=False)]
ax1.scatter(buys_df['date'], buys_df['portfolio_value'], color='#16a34a', s=25, marker='^', zorder=5, alpha=0.7, label='Buy')
ax1.scatter(sells_df['date'], sells_df['portfolio_value'], color='#dc2626', s=50, marker='v', zorder=5, label='Sell 70%')

ax1.set_title('BNB DCA Buy-the-Bear — Progressive Reinvestment (1% start, +1%/buy, cap 40%)', fontsize=13, fontweight='bold')
ax1.set_ylabel('USDT')
ax1.legend(loc='upper left', fontsize=8, ncol=3)
ax1.grid(True, alpha=0.25)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

ax2 = fig.add_subplot(gs[1])
ax2.fill_between(results['date'], 0, dd_raw * 100, color='#dc2626', alpha=0.35)
ax2.plot(results['date'], dd_raw * 100, color='#dc2626', linewidth=0.8)
ax2.axhline(y=max_dd, color='#991b1b', linestyle=':', linewidth=1, label=f'Max DD: {max_dd:.1f}%')
ax2.set_ylabel('Drawdown (%)')
ax2.set_ylim(bottom=0)
ax2.legend(loc='lower left', fontsize=9)
ax2.grid(True, alpha=0.25)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

ax3 = fig.add_subplot(gs[2])
ax3.plot(results['date'], results['close'], color='#f59e0b', linewidth=1.5)
ax3.fill_between(results['date'], 0, results['close'], color='#f59e0b', alpha=0.15)
ax3.set_ylabel('BNB Price (USDT)')
ax3.grid(True, alpha=0.25)
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

ax4 = fig.add_subplot(gs[3])
ax4.step(results['date'], results['reinvest_pct'], where='post', color='#8b5cf6', linewidth=1.5)
ax4.axhline(y=CAP, color='#8b5cf6', linestyle=':', alpha=0.5, label=f'Cap: {CAP}%')
ax4.set_ylabel('Reinvest Rate (%)')
ax4.set_xlabel('Date')
ax4.set_ylim(0, CAP * 1.15)
ax4.legend(loc='upper left', fontsize=9)
ax4.grid(True, alpha=0.25)
ax4.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig('dca-bt-buy-bot-bnb-final.png', dpi=150, bbox_inches='tight')
print('Chart saved as dca-bt-buy-bot-bnb-final.png')
plt.show()

In [ ]:
# Chart 2: BNB holdings over time

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

ax1.fill_between(results['date'], 0, results['coin_held'], color='#f59e0b', alpha=0.5, step='post')
ax1.step(results['date'], results['coin_held'], where='post', color='#d97706', linewidth=1.5)
ax1.set_ylabel('BNB Holdings')
ax1.set_title('BNB Accumulation Over Time')
ax1.grid(True, alpha=0.25)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

ax2.step(results['date'], results['reinvest_pct'], where='post', color='#8b5cf6', linewidth=1.5)
ax2.axhline(y=CAP, color='#8b5cf6', linestyle=':', alpha=0.5, label=f'Cap: {CAP}%')
ax2.set_ylabel('Reinvest Rate (%)')
ax2.set_xlabel('Date')
ax2.set_ylim(0, CAP * 1.15)
ax2.legend(loc='upper left', fontsize=9)
ax2.grid(True, alpha=0.25)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig('dca-bt-buy-bot-bnb-holdings.png', dpi=150, bbox_inches='tight')
print('Chart saved as dca-bt-buy-bot-bnb-holdings.png')
plt.show()

In [ ]:
# All trades table
pd.set_option('display.max_rows', None)
with_actions = results[results['action'] != 'nothing'].copy()
with_actions[['date', 'close', 'reinvest_pct', 'action', 'coin_held', 'coin_value', 'usdt_reserve', 'total_invested', 'portfolio_value']]